In [1]:
from loqs.backends import QSimQuantumState, PyGSTiNoiseModel
from loqs.core import QuantumProgram

from loqs.codepacks import d3_5q_code

import pygsti

In [2]:
code_5q = d3_5q_code.create_qec_code()

In [3]:
code_5q.instructions.keys()

dict_keys(['Non-FT Minus Prep', 'X', 'Z', 'H', 'Measure Physical Qubits'])

In [4]:
# Define a pyGSTi processor spec and noise model
# This is the only pyGSTi required here,
# and could eventually be traded out for something else
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]
gate_names = ["Gxpi2", "Gypi2", "Gzpi2", "Gh", "Gcnot", "Gcphase"]

# TODO: Currently Iz does not need to be set here
# This is because QSimQuantumState actually does not try to pull the rep
# Otherwise, this would result in a KeyError in PyGSTiNoiseModel.get_reps(),
# since it technically should be provided
pspec = pygsti.processors.QubitProcessorSpec(
    len(qubits), 
    gate_names=gate_names,
    qubit_labels=qubits,
    availability={k: "all-combinations" for k in gate_names}
)

ideal_model_pygsti = pygsti.models.create_crosstalk_free_model(pspec)

ideal_model = PyGSTiNoiseModel(ideal_model_pygsti)


In [5]:
# Logical quantum program information
patch_types = {"5Q": code_5q}

# Stack
# The "Init *" instructions will be generated by the QuantumProgram based on the values
# of state_type and patch_types below
stack = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}), # Autogenerated from state_tyep
    ("Init Patch 5Q", None, ("L0", qubits)), # Autogenerated from patch_types. Note that 5Q here must be a key
    ("Non-FT Minus Prep", "L0"),
    ("H", "L0"),
    ("Measure Physical Qubits", "L0")
]

program = QuantumProgram(
    stack,
    default_noise_model=ideal_model,
    state_type=QSimQuantumState,
    patch_types=patch_types,
    name="Test program"
)

In [6]:
print(stack)

[('Init State', None, (7,), {'qubit_labels': ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']}), ('Init Patch 5Q', None, ('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6'])), ('Non-FT Minus Prep', 'L0'), ('H', 'L0'), ('Measure Physical Qubits', 'L0')]


In [7]:
print(program.history[-1]["stack"])

InstructionStack with 5 items:
  InstructionLabel(Init State,None,(7,),{qubit_labels: ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
  InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
  InstructionLabel(Non-FT Minus Prep,L0,(),{})
  InstructionLabel(H,L0,(),{})
  InstructionLabel(Measure Physical Qubits,L0,(),{})



In [8]:
program.run()

Working on frame 1
Working on frame 2
Working on frame 3
Working on frame 4
Working on frame 5
Working on frame 6
Working on frame 7


In [10]:
print(program.history)

History with 8 items:
  Frame Adding InstructionStack from new QuantumProgram Test program (1 objects):
    'stack': 
      InstructionStack with 5 items:
        InstructionLabel(Init State,None,(7,),{qubit_labels: ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']})
        InstructionLabel(Init Patch 5Q,None,('L0', ['A0', 'A1', 'D2', 'D3', 'D4', 'D5', 'D6']),{})
        InstructionLabel(Non-FT Minus Prep,L0,(),{model: Physical pyGSTi noise model})
        InstructionLabel(H,L0,(),{model: Physical pyGSTi noise model,patch_label: L0,stack: InstructionStack with 1 items:
          InstructionLabel(Measure Physical Qubits,L0,(),{model: Physical pyGSTi noise model})})
        InstructionLabel(Measure Physical Qubits,L0,(),{model: Physical pyGSTi noise model})

  Frame QSimQuantumState state builder result (5 objects):
    'state': EXPIRED
    'instruction': 
      Instruction QSimQuantumState state builder
        Input parameter spec:
          InputSpec with 4 items:
            InputParam(0,[